# Картографирование статистических данных


В этом разделе мы рассмотрим, как можно визуализировать данные на примере Базы данных муниципальных образований (БДМО) Росстата


## Импортируем библиотеки


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mapclassify as mc

## Картографирование БДМО Росстата


### Загружаем границы муниципалитетов

Тут небольшой пример, границы всех муниципальных образований верхнего уровня можно скачать [тут](https://geoportal.hse.ru/portal/apps/sites/#/geodata/datasets/12e78b9c3bf04feea944f9d53eb3d079/about)

Если нужно учитывать также изменения границ муниципальных образований со временем, потребуется отдельный набор данных (с историей границ).

> 📥 Файл с данными для этого примера (`muni2.geojson`) можно скачать [из этой папки на Google Drive](https://drive.google.com/drive/folders/12gsEj8kOjDP2ww36wgjW2xAXNljIIMfV?usp=sharing) и положить в папку `data`.


In [ ]:
muni = gpd.read_file('data/muni2.geojson')

muni.head()

### Загружаем и обрабатываем статистические данные


- выбираем/рассчитываем подходящие показатели
- проверяем полноту данных за разные годы


Данные БДМО от проекта "Если быть точным":

- Полный набор данных + документация [тут](https://tochno.st/datasets/bdmo)
- Данные по отдельным показателям [тут](https://docs.google.com/spreadsheets/d/1Z66eXdgzyBg2NPV4cRzrTtv6BSZFVaAi/edit?pli=1&gid=1495188554#gid=1495188554)

> 📥 Файлы с данными для этого примера (`bdmo_pop_sample.csv` и `bdmo_birth_sample.csv`) можно скачать [из этой папки на Google Drive](https://drive.google.com/drive/folders/12gsEj8kOjDP2ww36wgjW2xAXNljIIMfV?usp=sharing) и положить в папку `data`.


Читаем данные о населении


In [ ]:
population = pd.read_csv('data/bdmo_pop_sample.csv', sep=",")

population.head()

Прежде чем строить карты, данные нужно **привести в удобный вид**. Сделаем два шага.

**Шаг 1. Оставляем только нужные строки.**
В колонке `mest` данные разбиты по типу территории: «Все население», «Городское население», «Сельское население». Для карты нам нужно всё население целиком, поэтому оставляем только строки, где `mest == 'Все население'` — иначе один и тот же муниципалитет попадёт в таблицу несколько раз (по строке на каждый тип).

**Шаг 2. Переводим таблицу из «длинного» формата в «широкий» с помощью `pivot_table`.**

Сейчас данные в **длинном** формате: для каждого муниципалитета — по одной строке на каждый год.

| oktmo    | year | value  |
| -------- | ---- | ------ |
| 14610000 | 2009 | 102185 |
| 14610000 | 2010 | 107994 |
| 14610000 | 2011 | 108925 |

С такими данными неудобно считать показатели и строить карты. Нам нужен **широкий** формат — одна строка на муниципалитет, а годы становятся отдельными столбцами:

| oktmo    | 2009   | 2010   | 2011   |
| -------- | ------ | ------ | ------ |
| 14610000 | 102185 | 107994 | 108925 |

Именно это и делает `pivot_table`:

- `index=[...]` — что оставляем в строках (описание муниципалитета: `oktmo`, регион, название и т.д.);
- `columns='year'` — какой столбец «раскладываем» в шапку таблицы (годы);
- `values='value'` — какие числа поставить на пересечении строки и столбца (значение показателя).

`.reset_index()` в конце возвращает поля из `index` обратно в обычные столбцы (иначе они станут «индексом» таблицы и с ними будет сложнее работать дальше).


In [ ]:
population_filtered = population[population['mest'] == 'Все население']

population_wide = population_filtered.pivot_table(index=['oktmo', 'indicator', 'unit', 'region', 'munr', 'municipality'], columns='year', values='value').reset_index()

population_wide.head()

Смотрим на кол-во отсутствующих данных (чтобы понять, какие годы не стоит рассматривать)


In [ ]:
missing_values = population_wide.isna().sum()
missing_values 

Читаем данные о рождаемости


In [ ]:
birth = pd.read_csv('data/bdmo_birth_sample.csv', sep=",")

birth.head()


In [ ]:
birth_wide = birth.pivot_table(index=['oktmo', 'indicator', 'unit', 'region', 'munr', 'municipality'], columns='year', values='value').reset_index()

birth_wide.head()

In [ ]:
missing_values_birth = birth_wide.isna().sum()
missing_values_birth

### Объединяем данные на основе ОКТМО

**ОКТМО** — код территории по общероссийскому классификатору. Он есть и в геоданных (границы), и в статистике, поэтому именно по нему мы сшиваем таблицы: к каждому муниципалитету подтягиваем его показатели.


Само объединение выполняет метод `.merge()`. Разберём важные детали:

- **`astype(str)`** — сначала приводим `oktmo` к строке во всех таблицах. Если в одной таблице код записан как число, а в другой как текст, Python посчитает `14610000` и `'14610000'` разными значениями, и строки не соединятся.
- **`left_on` / `right_on`** — указываем, по какому столбцу сопоставлять строки (в обеих таблицах это `oktmo`).
- **`how='left'`** — «левое» объединение: берём **все** муниципалитеты из границ (`muni`) и подтягиваем к ним статистику. Если для какого-то муниципалитета статистики нет, строка всё равно останется, а в столбцах со статистикой будет пусто (`NaN`). Так мы не теряем ни одной территории на карте.
- **`suffixes=('', '_birth')`** — в таблицах о населении и о рождаемости столбцы называются одинаково (`2016`, `2017`, ...). Чтобы при склейке они не перезаписали друг друга, ко второму набору добавляется суффикс: `2016` (население) и `2016_birth` (рождаемость).

Объединяем в два приёма: сначала границы + население, затем результат + рождаемость.


In [ ]:
# Привести оба поля к строковому типу
muni['oktmo'] = muni['oktmo'].astype(str)
population_wide['oktmo'] = population_wide['oktmo'].astype(str)
birth_wide['oktmo'] = birth_wide['oktmo'].astype(str)


muni_stat_pop = muni.merge(population_wide, 
                      left_on='oktmo',
                      right_on='oktmo', 
                      how='left',
                      suffixes=('', '_pop'))


muni_stat_all = muni_stat_pop.merge(birth_wide, 
                      left_on='oktmo',
                      right_on='oktmo', 
                      how='left',
                      suffixes=('', '_birth'))


muni_stat_all.head()


### Рассчитываем коэффициент рождаемости в 2016 и 2020 годах


Теперь на основе объединённых данных рассчитаем **коэффициент рождаемости**, который будем картографировать.

Делим число родившихся на численность населения и умножаем на 1000:

$$\text{коэффициент рождаемости} = \frac{\text{число родившихся}}{\text{численность населения}} \times 1000$$

Умножение на 1000 переводит долю в **промилле (‰)** — то есть показывает, сколько детей родилось на каждую тысячу жителей. Так удобнее сравнивать территории разного размера: абсолютное число рождений в большом городе и маленьком районе несопоставимо, а «сколько рождений на 1000 человек» — уже корректное сравнение.

Считаем коэффициент отдельно для 2016 и 2020 годов — потом сравним их на картах.


In [ ]:
muni_stat_all['natincrate2016'] = (muni_stat_all['2016_birth'] / muni_stat_all['2016']) * 1000
muni_stat_all['natincrate2020'] = (muni_stat_all['2020_birth'] / muni_stat_all['2020']) * 1000

Построим карту коэффициента рождаемости за 2020 год:


In [ ]:
ax = muni_stat_all.plot(column='natincrate2020', cmap='BuPu', linewidth=0.01, edgecolor='0.75', legend=True, scheme="naturalBreaks")

# добавляем название
ax.set_title('Коэффициент рождаемости, ‰')

# скрываем координатные оси
ax.axis('off')

plt.show()

Теперь сравним коэффициент рождаемости за **2016 и 2020 годы**. Сначала построим две карты, где для каждого года интервалы классификации (`naturalBreaks`) считаются **отдельно**:


In [ ]:

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# 2016 год
plot1 = muni_stat_all.plot(
    column='natincrate2016',
    cmap='BuPu',
    linewidth=0.01, 
    edgecolor='0.75',
    legend=True,
    scheme="naturalBreaks",
    ax=ax1
)
ax1.set_title('Коэффициент рождаемости, 2016, ‰')
ax1.axis('off')

# 2020 год
plot2 = muni_stat_all.plot(
    column='natincrate2020',
    cmap='BuPu',
    linewidth=0.01,
    edgecolor='0.75', 
    legend=True,
    scheme="naturalBreaks",
    ax=ax2
)
ax2.set_title('Коэффициент рождаемости, 2020, ‰')
ax2.axis('off')

plt.tight_layout()
plt.show()

У карт выше есть подвох: интервалы (а значит, и легенда) у каждого года **свои**, поэтому напрямую сравнивать цвета между картами нельзя. Чтобы сравнение было корректным, зададим **единые интервалы** для обоих годов и построим карты заново:


In [ ]:
# Вычисляем общие интервалы на основе данных за оба года
all_values = np.concatenate([
    muni_stat_all['natincrate2016'].dropna().values,
    muni_stat_all['natincrate2020'].dropna().values
])

# Создаем классификатор с общими интервалами (k=5 -> 5 интервалов)
classifier = mc.NaturalBreaks(all_values, k=5)
common_bins = classifier.bins

print("Общие интервалы для обеих карт:", common_bins)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# 2016 год — с общими интервалами
muni_stat_all.plot(
    column='natincrate2016',
    cmap='BuPu',
    linewidth=0.01,
    edgecolor='0.75',
    legend=True,
    scheme="User_Defined",                       # пользовательские интервалы
    classification_kwds={'bins': common_bins},   # задаём общие интервалы
    ax=ax1
)
ax1.set_title('Коэффициент рождаемости, 2016, ‰')
ax1.axis('off')

# 2020 год — те же самые интервалы
muni_stat_all.plot(
    column='natincrate2020',
    cmap='BuPu',
    linewidth=0.01,
    edgecolor='0.75',
    legend=True,
    scheme="User_Defined",
    classification_kwds={'bins': common_bins},
    ax=ax2
)
ax2.set_title('Коэффициент рождаемости, 2020, ‰')
ax2.axis('off')

plt.tight_layout()
plt.show()

## Итог

На примере **БДМО Росстата** мы прошли полный путь картографирования статистических данных: загрузили границы муниципалитетов, подготовили и проверили статистику по годам, сшили таблицы с геометрией по **ОКТМО** и построили карты коэффициента рождаемости.

Главное, что стоит запомнить: чтобы карты за разные годы можно было **сравнивать между собой**, интервалы классификации должны быть **общими** — иначе одинаковые цвета на разных картах будут означать разные значения.

Успехов!
